In [ ]:
import os
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

class AsistenteMetroRAG:
    def __init__(self):
        load_dotenv()
        self.client = OpenAI(
            base_url=os.environ.get("GITHUB_BASE_URL"), 
            api_key=os.environ.get("GITHUB_TOKEN")
        )
        self.documentos = [
            "Tarifa PUNTA (07:00-08:59 y 18:00-19:59): Cuesta $840. Es el horario de la mañana y tarde con más gente.",
            "Tarifa VALLE (09:00-17:59 y 20:00-20:44): Cuesta $760. Horario intermedio.",
            "Tarifa BAJO (06:00-06:59 y 20:45-23:00): Cuesta $680. Horario temprano o muy tarde.",
            "Para ir a PUENTE ALTO: Debes tomar la Línea 4 (Azul) que llega hasta Plaza de Puente Alto.",
            "COMBINACIÓN: En la estación Tobalaba puedes cambiar entre Línea 1 y Línea 4 para ir a Puente Alto."
        ]
        
        self.embeddings_db = []
        self._preparar_rag()

    def _get_embedding(self, texto):
        response = self.client.embeddings.create(
            input=texto,
            model="text-embedding-3-small"
        )
        return response.data[0].embedding

    def _preparar_rag(self):
        print("Indexando base de conocimientos...")
        for doc in self.documentos:
            self.embeddings_db.append(self._get_embedding(doc))

    def _recuperar_contexto(self, consulta_usuario, top_k=4):
        v_consulta = self._get_embedding(consulta_usuario)
        similitudes = [np.dot(v_consulta, v_db) for v_db in self.embeddings_db]
        indices = np.argsort(similitudes)[-top_k:]
        return "\n".join([self.documentos[i] for i in indices])

    def consultar_base_conocimiento(self, consulta_usuario, top_k=4):
        """Tool de RAG: recupera los fragmentos mas relevantes desde la base
        de conocimientos usando embeddings + similitud de coseno."""
        return self._recuperar_contexto(consulta_usuario, top_k)

    def consultar(self, pregunta):
        contexto = self.consultar_base_conocimiento(pregunta)
        prompt_sistema = f"""
        Eres el asistente oficial de Metro de Santiago. 
        Usa SOLO la siguiente informacion recuperada para responder:
        {contexto}

        Responde siempre en formato JSON con estas llaves:
        - 'metacognicion': Tu razonamiento logico.
        - 'respuesta': La respuesta final al usuario.
        - 'fuentes_usadas': Que fragmentos del contexto utilizaste.
        """

        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": pregunta}
            ],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

if __name__ == "__main__":
    asistente = AsistenteMetroRAG()
    # Demo de consultar_base_conocimiento (tool RAG)
    contexto = asistente.consultar_base_conocimiento("Cuanto cuesta el pasaje a las 8 am?")
    print("=== Contexto recuperado por consultar_base_conocimiento ===")
    print(contexto)
    print()
    # Demo completa del pipeline RAG
    resultado = asistente.consultar("Cuanto me sale el pasaje a las 8 am y como llego a Puente Alto?")
    print("=== Respuesta final del asistente ===")
    print(json.dumps(resultado, indent=4, ensure_ascii=False))